# Vorbereitung des Teilexperiment B

Aufbereitung der Daten für den zweiten Experimentendurchlauf. Verwendung des Dataset vrdu ad-buy-form gemischt mit dem DocILE Korpus zur externen Validierung der Experimentbedingungen auf Daten, die einen echten Einsatz simulieren.

In [19]:
import os

RAW_DATA_PATH_DOCILE = "../data/raw/docile"
OUTPUT_DIR = "../data/processed/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [20]:
import json
import pandas as pd
from pathlib import Path


def create_dataframe_from_json(json_path, pdf_dir, ocr_dir, ann_dir):
    with open(RAW_DATA_PATH_DOCILE + json_path, "r") as f:
        ids = json.load(f)

    data_list = []

    for file_id in ids:
        pdf_file = Path(RAW_DATA_PATH_DOCILE + pdf_dir) / f"{file_id}.pdf"
        ocr_file = Path(RAW_DATA_PATH_DOCILE + ocr_dir) / f"{file_id}.json"
        ann_file = Path(RAW_DATA_PATH_DOCILE + ann_dir) / f"{file_id}.json"

        if ocr_file.exists() and ann_file.exists():
            with open(ocr_file, "r") as f_ocr:
                ocr_content = json.load(f_ocr)
            with open(ann_file, "r") as f_ann:
                ann_content = json.load(f_ann)

            data_list.append(
                {
                    "doc_id": file_id,
                    "file_path": str(pdf_file),
                    "ocr": ocr_content,
                    "annotations": ann_content,
                }
            )

    return pd.DataFrame(data_list)


df_docile = create_dataframe_from_json("/train.json", "/pdfs", "/ocr", "/annotations")

In [21]:
df_docile.head()

,doc_id,file_path,ocr,annotations
0,00134dd365a24343b35b78c6,../data/raw/docile/pdfs/00134dd365a24343b35b78...,"{'pages': [{'page_idx': 0, 'dimensions': [1616...",{'field_extractions': [{'bbox': [0.78253089891...
1,00136a27c7774c1e8dc6b2f2,../data/raw/docile/pdfs/00136a27c7774c1e8dc6b2...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.63452136714...
2,002e3cf97973428f905671b3,../data/raw/docile/pdfs/002e3cf97973428f905671...,"{'pages': [{'page_idx': 0, 'dimensions': [1689...",{'field_extractions': [{'bbox': [0.72741935483...
3,002f9b82b74f4258b3b072d0,../data/raw/docile/pdfs/002f9b82b74f4258b3b072...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.49176869947...
4,003568b1286f4dab953fc2d5,../data/raw/docile/pdfs/003568b1286f4dab953fc2...,"{'pages': [{'page_idx': 0, 'dimensions': [1684...",{'field_extractions': [{'bbox': [0.82242516938...


### Preprocess both Datasets

In [22]:
def ocr_to_text_docile(daten):
    clean_text = ""
    for page in daten.get("pages", []):
        for block in page.get("blocks", []):
            for line in block.get("lines", []):
                worte = [word.get("value", "") for word in line.get("words", [])]
                clean_text += " ".join(worte) + "\n"
            clean_text += "\n"
    return clean_text

df_docile["ocr_text"] = df_docile["ocr"].apply(ocr_to_text_docile)

In [23]:
def extract_ground_truth_docile(annotations):
    ground_truth = {}
    if isinstance(annotations, dict):
        extractions = annotations.get("field_extractions", [])
        for field in extractions:
            fieldtype = field.get("fieldtype")
            text = field.get("text", "").strip()

            if fieldtype in ground_truth:
                ground_truth[fieldtype] += " " + text
            else:
                ground_truth[fieldtype] = text

    return ground_truth


df_docile["ground_truth"] = df_docile["annotations"].apply(extract_ground_truth_docile)

In [24]:
df_docile.head(5)

,doc_id,file_path,ocr,annotations,ocr_text,ground_truth
0,00134dd365a24343b35b78c6,../data/raw/docile/pdfs/00134dd365a24343b35b78...,"{'pages': [{'page_idx': 0, 'dimensions': [1616...",{'field_extractions': [{'bbox': [0.78253089891...,"WIRE INSTRUCTIONS:\nNationsBank, N.A.\nBaltimo...","{'document_id': '990304502', 'date_issue': '03..."
1,00136a27c7774c1e8dc6b2f2,../data/raw/docile/pdfs/00136a27c7774c1e8dc6b2...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.63452136714...,Please Remit To:\nKMOZ 92.3 The Moose\n1360 E....,"{'document_id': '17685-2 17685-2', 'date_issue..."
2,002e3cf97973428f905671b3,../data/raw/docile/pdfs/002e3cf97973428f905671...,"{'pages': [{'page_idx': 0, 'dimensions': [1689...",{'field_extractions': [{'bbox': [0.72741935483...,"UMI PUBLICATIONS, INC.\nP.O.BOX 30036 (28230)\...","{'document_id': '096084', 'date_issue': '12/09..."
3,002f9b82b74f4258b3b072d0,../data/raw/docile/pdfs/002f9b82b74f4258b3b072...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.49176869947...,Page 1 of1\n\nINVOICE\nInvoice #\n2081683-1\nS...,"{'document_id': '2081683-1', 'date_issue': '09..."
4,003568b1286f4dab953fc2d5,../data/raw/docile/pdfs/003568b1286f4dab953fc2...,"{'pages': [{'page_idx': 0, 'dimensions': [1684...",{'field_extractions': [{'bbox': [0.82242516938...,Appud 10032089\n\nWAIT\nKLEN\nSAINISIC\nWalt K...,"{'document_id': '23299 23299', 'date_issue': '..."


In [25]:
def count_docile_annotations(annotations_dict):
    if not isinstance(annotations_dict, dict):
        return 0

    count = len(annotations_dict.get("field_extractions", []))

    # Berücksichtigt auch Tabellenzeilen, falls diese separat in DOCILE existieren
    count += len(annotations_dict.get("line_item_extractions", []))

    return count


df_docile["annotation_count"] = df_docile["annotations"].apply(count_docile_annotations)

p33 = df_docile["annotation_count"].quantile(0.33)
p66 = df_docile["annotation_count"].quantile(0.66)

df_docile["complexity"] = df_docile["annotation_count"].apply(
    lambda c: "L1" if c <= p33 else ("L2" if c <= p66 else "L3")
)

print(df_docile["complexity"].value_counts())
print(f"\nTerzilgrenzen: L1 <= {p33:.0f}, L2 <= {p66:.0f}, L3 > {p66:.0f}")

complexity
L1    1878
L3    1731
L2    1571
Name: count, dtype: int64

Terzilgrenzen: L1 <= 19, L2 <= 38, L3 > 38


# Auswertung des Splits

### Ziehung der Stichprobe

## Statistische Auswertung der Stichprobenziehung

### Verteilung der Daten


### Statistischer Test der Unterschiede zwischen Annotationen in den Gruppen


### Prüfung der relativen Informationsdichte

### Prüfung der Anzahl von line_items pro Komplexität

### Erstellung des Korpus für Teilexperiment B

In [10]:
experiment_documents = {}

In [11]:
with open("../data/processed/corpus_experiment_B_external.jsonl", "w") as f:
    for entry in experiment_documents:
        f.write(entry.to_json(ensure_ascii=False) + "\n")